<a href="https://colab.research.google.com/github/kodomotachi/heartify-AI/blob/main/Food_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -U opik langgraph langchain langchain-groq langchain-pinecone langchain-huggingface sentence-transformers pinecone-client

# Task
Create a personalized nutrition recommendation system using LangGraph and LangChain, utilizing Groq's `llama-3.3-70b-versatile` model for analysis and generation. The workflow should process blood test results and user preferences to provide validated dietary advice.

In [ ]:
# Library Loading
import os
from typing import TypedDict, List, Dict, Any
from google.colab import userdata

# LangChain Core & Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser # Gom lại

# Vector DB & Embeddings
from pinecone import Pinecone, ServerlessSpec # Dùng để quản lý Index (Admin tasks)
from langchain_pinecone import PineconeVectorStore # Dùng để Search/Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

# LangGraph & Monitoring
from langgraph.graph import StateGraph, START, END
from opik.integrations.langchain import OpikTracer

import hashlib

## Configure Groq API Key

### Subtask:
Securely set the `GROQ_API_KEY` environment variable for authentication.


In [ ]:
# Environment Loading
def load_env_var(key, required=True):
    """Loads a key from Colab userdata to os.environ."""
    try:
        os.environ[key] = userdata.get(key)
        return True
    except Exception:
        if required:
            print(f"Error: Required secret '{key}' not found.")
        return False

# Configure standard keys
load_env_var("GROQ_API_KEY")
load_env_var("PINECONE_API_KEY")

# Configure Opik for observability
if load_env_var("OPIK_API_KEY", required=False):
    os.environ["OPIK_WORKSPACE"] = "phuc-duy-loc-nguyen" # Optional: Change if you have a specific workspace
    os.environ["OPIK_PROJECT_NAME"] = "FOOD"
    print("Opik configured successfully.")
else:
    print("Warning: OPIK_API_KEY not found in secrets. Tracing might not work.")

Opik configured successfully.


## Define Graph State

### Subtask:
Define the `TypedDict` structure for the graph state.


In [ ]:
class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        user_health_record: user input blood test record analysis
        user_preferences: user input dietary preferences
        search_queries: list of generated search queries
        food_items: list of retrieved food item content
        recommendation: final generated dietary advice
    """
    user_health_record: str
    user_preferences: str
    search_queries: List[str]
    food_items: List[Dict[str, Any]]
    recommendation: str

print("GraphState class defined.")

GraphState class defined.


## Implement Diagnosis Node

### Subtask:
Implement the diagnosis node using ChatGroq to analyze user inputs and generate search queries.


In [ ]:
# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Define the prompt template optimized for single semantic search query
diagnosis_prompt = PromptTemplate(
    template="""You are a semantic search query generator for a recipe nutrition database.

BLOOD TEST RESULTS:
{user_health_record}

USER PREFERENCES:
{user_preferences}

DATABASE STRUCTURE:
Each recipe is stored as: "Recipe Name. Nutrition: X calories, Xg protein, Xg fat, Xg carbohydrates, Xg fiber. Ingredients: [list]. Diet: [labels]. Health: [labels]. Meal: [type]. Cuisine: [type]"

QUERY GENERATION RULES:
1. Create ONE comprehensive query that addresses all health concerns from the blood test
2. Combine 5-8 relevant terms into a single query string
3. Use specific nutritional attributes: "high-protein", "low-fat", "high-fiber", "low-calorie"
4. Include key ingredient names: "salmon", "quinoa", "spinach", "chickpeas", "avocado"
5. Add relevant diet labels: "vegan", "low-carb", "gluten-free", "dairy-free", "plant-based"
6. Add health labels: "heart-healthy", "low-cholesterol", "iron-rich", "omega-3-rich"
7. NO conversational phrases like "recipes for", "foods to eat", "meals that"

EXAMPLES OF GOOD QUERIES:
- "salmon omega-3 high-protein low-cholesterol heart-healthy Mediterranean"
- "quinoa chickpeas vegan high-fiber iron-rich plant-based"
- "spinach kale leafy-greens iron-rich low-calorie high-fiber low-sodium"
- "lean-protein grilled low-fat gluten-free heart-healthy dinner"

BAD QUERIES (avoid):
- "fish recipes for lowering cholesterol"
- "what foods help with iron"

Generate ONE comprehensive search query that combines the most important health needs.

Return as JSON:
{{
  "search_query": "your single query here"
}}

Do not include any other text outside the JSON.""",
    input_variables=["user_health_record", "user_preferences"]
)


# Create the chain
diagnosis_chain = diagnosis_prompt | llm | JsonOutputParser()


In [ ]:
def diagnosis_node(state):
    """
    Analyzes blood test and preferences to generate a single semantic search query.
    Refactored for robustness and performance.
    """
    print("---DIAGNOSIS NODE---")

    # Use .get() for safer access
    user_health_record = state.get('user_health_record', '')
    user_preferences = state.get('user_preferences', '')

    # Define forbidden phrases as a set for O(1) lookups
    FORBIDDEN_PHRASES = {
        "recipe", "food", "meal", "dish", "how to", "what is", "help with", "for"
    }

    try:
        # Invoke the chain
        result = diagnosis_chain.invoke({
            "user_health_record": user_health_record,
            "user_preferences": user_preferences
        })

        # Extract and normalize the query
        search_query = result.get("search_query", "").strip().lower()

        # Filter out conversational filler words
        cleaned_words = [
            w for w in search_query.split()
            if w not in FORBIDDEN_PHRASES
        ]
        cleaned_query = " ".join(cleaned_words)

        # Validation: Ensure query has sufficient terms (at least 3)
        if len(cleaned_words) < 3:
            print(f"WARNING: Query may be too short: '{cleaned_query}'. Reverting to original.")
            cleaned_query = search_query

        print(f"Generated Query: {cleaned_query}")

        # Return as single-item list
        return {"search_queries": [cleaned_query]}

    except Exception as e:
        print(f"Error generating query: {e}")
        return {"search_queries": []}

print("Diagnosis Node implemented.")

Diagnosis Node implemented.


## Implement Retrieve and Rerank Nodes

### Subtask:
Implement the retrieval logic to fetch relevant food items from a simulated knowledge base.


In [ ]:
pinecone_index_name = "food-nutrition-recipes"

# Dùng wrapper của LangChain thay vì SentenceTransformer gốc
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print(f"Connecting to Pinecone index '{pinecone_index_name}'...")
vectorstore = PineconeVectorStore(
    index_name=pinecone_index_name,
    embedding=embeddings,
)
print("Pinecone Vector Database integrated.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Connecting to Pinecone index 'food-nutrition-recipes'...
Pinecone Vector Database integrated.


In [ ]:
# Initialize Pinecone Index directly for the tool
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
index = pc.Index(pinecone_index_name)

def search_safe_foods(query_text, user_allergens=[], top_k=5,
                      min_protein=None, max_calories=None):
    """
    Search for safe foods with allergen and nutrition filtering
    Adapted to use the existing 'embeddings' object.
    """
    # Create embedding using the existing LangChain embeddings object
    query_embedding = embeddings.embed_query(query_text)

    # Build metadata filter
    filters = []

    # Allergen filters
    if user_allergens:
        for allergen in user_allergens:
            # Metadata has fields like 'has_dairy', 'has_egg', etc.
            filters.append({f"has_{allergen}": {"$eq": False}})

    # Nutrition filters
    if min_protein is not None:
        # Metadata uses 'protein_g'
        filters.append({"protein_g": {"$gte": min_protein}})

    if max_calories is not None:
        filters.append({"calories": {"$lte": max_calories}})

    # Combine filters with AND logic
    metadata_filter = {"$and": filters} if filters else None

    # Query Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=metadata_filter
    )

    # Format results
    foods = []
    for match in results['matches']:
        m = match['metadata']
        foods.append({
            'name': m.get('recipe_name', 'Unknown'), # Mapped from recipe_name
            'similarity_score': round(match['score'], 3),
            'calories': m.get('calories', 'N/A'),
            'protein': m.get('protein_g', 'N/A'),    # Mapped from protein_g
            'fat': m.get('fat_g', 'N/A'),            # Mapped from fat_g
            'carbs': m.get('carbohydrates_g', 'N/A'), # Mapped from carbohydrates_g
            'fiber': m.get('fiber_g', 'N/A'),        # Mapped from fiber_g
            'allergens': m.get('allergen_list', []), # Mapped from allergen_list
            'nutrition_density': m.get('nutrition_density', 0),
            'url': m.get('url', 'N/A') # URL
        })

    return foods

In [ ]:
def retrieve_node(state):
    """
    Retrieves relevant food items using advanced search with reranking.
    Returns a structured list of dictionaries, including a hashed ID for secure/clean referencing.
    """
    print("---RETRIEVE NODE---")
    search_queries = state.get('search_queries', [])
    user_preferences = state.get('user_preferences', "").lower()

    # --- 1. Enhanced Allergen Detection ---
    allergen_keywords = {
        'dairy': ['cheese', 'cream', 'milk', 'butter', 'ricotta', 'mozzarella',
                  'cheddar', 'parmesan', 'gouda', 'brie', 'feta'],
        'egg': ['egg', 'omelet', 'scrambled'],
        'peanuts': ['peanut'],
        'tree_nuts': ['tahini', 'hazelnut', 'almond', 'cashew', 'walnut', 'pecan'],
        'shellfish': ['shrimp', 'crab', 'lobster', 'clam', 'mussel'],
        'fish': ['fish', 'salmon', 'tuna', 'cod'],
        'gluten': ['bread', 'croissant', 'biscuit', 'pancake', 'muffin',
                    'burrito', 'tamale', 'roll', 'wheat', 'barley', 'rye'],
        'soy': ['tofu', 'soy', 'edamame']
    }

    detected_allergens = set()
    if "vegan" in user_preferences:
        detected_allergens.update(['dairy', 'egg', 'fish', 'shellfish'])

    for category, keywords in allergen_keywords.items():
        if f"no {category}" in user_preferences or f"allergic to {category}" in user_preferences:
            detected_allergens.add(category)
        for keyword in keywords:
            if f"no {keyword}" in user_preferences or f"allergic to {keyword}" in user_preferences:
                detected_allergens.add(category)

    detected_allergens_list = list(detected_allergens)
    if detected_allergens_list:
        print(f"Applying allergen filters: {detected_allergens_list}")

    # --- 2. Retrieval & Reranking ---

    # Store results as a list of dictionaries (List[Dict[str, Any]])
    food_items_list: List[Dict[str, Any]] = []
    seen_ids = set() # To track unique food items by name

    for query in search_queries:
        # Step A: Vector search
        results = search_safe_foods(
            query,
            user_allergens=detected_allergens_list,
            top_k=30
        )

        # [Optimization] Create a lookup map for O(1) access
        results_map = {item['name']: item for item in results}

        # Step B: Prepare documents for reranking
        documents = [
            {
                "id": item['name'],
                "text": f"{item['name']}. Calories: {item['calories']}, Protein: {item['protein']}g, Fat: {item['fat']}g"
            }
            for item in results
        ]

        # Step C: Rerank if results exist
        if documents:
            reranked = pc.inference.rerank(
                model="bge-reranker-v2-m3",
                query=query,
                documents=documents,
                top_n=10,
                return_documents=True
            )

            # Step D: Extract & Structure Results
            for ranked_doc in reranked.get('data', []):
                food_name = ranked_doc['document']['id']

                # Retrieve full item details from map
                matching_item = results_map.get(food_name)

                # Ensure uniqueness
                if matching_item and food_name not in seen_ids:

                    url = matching_item.get('url', '')

                    # --- HASH ID LOGIC ---
                    # Generates a unique ID based on the URL hash
                    if url:
                        hash_id = hashlib.sha256(url.encode('utf-8')).hexdigest()
                    else:
                        # Fallback id if URL is missing (using name instead)
                        hash_id = hashlib.sha256(food_name.encode('utf-8')).hexdigest()[:16]

                    # Create a structured dictionary object
                    structured_item = {
                        "id": hash_id,   # <--- Added Hash ID field
                        "name": matching_item['name'],
                        # 'content': Text optimized for LLM prompting (includes ID for referencing)
                        "content": (
                            f"Food: {matching_item['name']}, "
                            f"Calories: {matching_item['calories']}, "
                            f"Protein: {matching_item['protein']}g, "
                            f"Allergens: {matching_item['allergens']}"
                        ),
                        "url": url,                          # URL kept separately
                        "score": ranked_doc.get('score', 0)  # Reranker score kept separately
                    }

                    food_items_list.append(structured_item)
                    seen_ids.add(food_name)

    if not food_items_list:
        print("No matches found via vector search and reranking.")
    else:
        print(f"Retrieved and reranked {len(food_items_list)} unique items.")
        if len(food_items_list) > 0:
            print(f"Sample item keys: {food_items_list[0].keys()}")

    # Return under 'food_items' key to match GraphState schema
    return {"food_items": food_items_list}


## Implement Generation Node

### Subtask:
Develop the generation node using `ChatGroq` to draft structured recommendations based on retrieved items and user context.


In [ ]:
generation_prompt = PromptTemplate(
    template="""You are an expert nutritionist creating evidence-based dietary recommendations.

BLOOD TEST RESULTS:
{user_health_record}

USER PREFERENCES:
{user_preferences}

RETRIEVED FOOD RECOMMENDATIONS (ranked by relevance to health needs):
{food_items}

INSTRUCTIONS:
1. **General Advice (1 Sentence)**: Provide a single, powerful sentence summarizing the core dietary strategy (e.g., "Focus on increasing iron intake through leafy greens while reducing saturated fats to lower cholesterol").
2. **Prioritize high-relevance items**: Focus your recommendations on foods with the highest Relevance scores from the list.

OUTPUT STRUCTURE:

## Strategic Advice
[Insert your 1-sentence general advice here]

## Food Recommendations
[Based on blood test, identify food types to eat]

CRITICAL RULES:
- Recommend ONLY foods from the "Retrieved Food Recommendations" list above, and display each item in bullet points (only 3 highest)
- Do not use technical words (like relevance score, similarity search), instead use user-friendly words
- Do not invent or suggest foods not in the provided list
""",
    input_variables=["user_health_record", "user_preferences", "food_items"]
)

generation_chain = generation_prompt | llm | StrOutputParser()

In [ ]:
def generation_node(state):
    """
    Generates dietary recommendations using structured food items.
    Handles sorting, context creation for LLM, and appending source URLs.
    """
    print("---GENERATION NODE---")

    # Get inputs from state
    user_health_record = state.get('user_health_record', '')
    user_preferences = state.get('user_preferences', '')
    # food_items is now a List[Dict], not List[str]
    food_items_list = state.get('food_items', [])

    formatted_context_for_llm = ""
    source_links = []

    # Check if we have valid items
    if isinstance(food_items_list, list) and len(food_items_list) > 0:

        # --- 1. SORT ITEMS ---
        # Sort directly by the 'score' key (no parsing needed!)
        # Handle cases where score might be missing using .get()
        food_items_list.sort(key=lambda x: x.get('score', 0.0), reverse=True)

        # --- 2. FORMAT CONTEXT & PREPARE LINKS ---
        context_lines = []

        for i, item in enumerate(food_items_list, 1):
            # Create a clean text line for the LLM to read
            # Example: "[1] Food: Salmon, Calories: 200..."
            line = f"[{i}] {item.get('content', 'Unknown Food')}"
            context_lines.append(line)

            # Store the URL separately to append later
            # Example: "[1] Salmon: http://..."
            url = item.get('url', 'N/A')
            name = item.get('name', 'Unknown')
            source_links.append(f"[{i}] {name}: {url}")

        # Join lines for the prompt
        formatted_context_for_llm = "\n".join(context_lines)

    else:
        formatted_context_for_llm = "No specific food items found matching constraints."

    # --- 3. INVOKE LLM CHAIN ---
    try:
        recommendation_text = generation_chain.invoke({
            "user_health_record": user_health_record,
            "user_preferences": user_preferences,
            # Only send the text content to the LLM (saves tokens)
            "food_items": formatted_context_for_llm
        })
    except Exception as e:
        print(f"Error invoking generation chain: {e}")
        recommendation_text = "Sorry, I could not generate a recommendation at this time."

    # --- 4. APPEND URLS TO OUTPUT ---
    # Combine the LLM's advice with the hard-coded source links
    if source_links:
        final_output = (
            f"{recommendation_text}\n\n"
            f"--- **Reference Links** ---\n"
            f"{chr(10).join(source_links)}" # chr(10) is newline
        )
    else:
        final_output = recommendation_text

    print(f"Generated Recommendation:\n{final_output[:300]}...")

    return {"recommendation": final_output}

print("Generation Node updated for structured data.")

Generation Node updated for structured data.


## Build and Run Workflow

### Subtask:
Construct the LangGraph workflow, compile it, and run a test case to demonstrate the complete system.


In [ ]:
# Initialize the graph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("diagnosis", diagnosis_node)
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generation_node)

# Define edges
workflow.add_edge(START, "diagnosis")
workflow.add_edge("diagnosis", "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the graph
app = workflow.compile()

# Define test inputs
inputs = {
    "user_health_record": "High cholesterol, Low Iron, Normal blood sugar.",
    "user_preferences": "fish"
}

# Initialize Opik Tracer
opik_tracer = OpikTracer(graph=app.get_graph(xray=True))

# Execute the workflow with tracing
print("---STARTING WORKFLOW WITH OPIK TRACING---")
try:
    final_state = app.invoke(inputs, config={"callbacks": [opik_tracer]})

    print("\n---WORKFLOW COMPLETE---")
    print(f"Final Recommendation:\n{final_state['recommendation']}")
    print("\nCheck your Opik dashboard to see the traces!")
except Exception as e:
    print(f"An error occurred execution: {e}")

---STARTING WORKFLOW WITH OPIK TRACING---
---DIAGNOSIS NODE---
Generated Query: chicken high-protein low-calorie high-fiber low-fat heart-healthy diabetes-friendly
---RETRIEVE NODE---
Retrieved and reranked 9 unique items.
Sample item keys: dict_keys(['id', 'name', 'content', 'url', 'score'])
---GENERATION NODE---
Generated Recommendation:
## Strategic Advice
To manage type 2 diabetes and excessive thirst, focus on consuming nutrient-dense, high-protein foods like chicken that are low in calories and allergens, while staying hydrated with plenty of water.

## Food Recommendations
Based on your blood test results and preferences, consi...

---WORKFLOW COMPLETE---
Final Recommendation:
## Strategic Advice
To manage type 2 diabetes and excessive thirst, focus on consuming nutrient-dense, high-protein foods like chicken that are low in calories and allergens, while staying hydrated with plenty of water.

## Food Recommendations
Based on your blood test results and preferences, consider the

In [ ]:
# Iterate through all keys in the final state and print them
for key, value in final_state.items():
    print(f"--- {key.upper()} ---")
    print(value)
    print("\n")

--- USER_HEALTH_RECORD ---
High cholesterol, Low Iron, Normal blood sugar.


--- USER_PREFERENCES ---
fish


--- SEARCH_QUERIES ---
['salmon high-protein low-fat omega-3-rich heart-healthy iron-rich high-fiber dairy-free']


--- FOOD_ITEMS ---
[{'id': 'dacdbf3c1e3f688b3200b0484f208452e4cfad06c9b677a3fe3c08eb6116037f', 'name': 'Middle Eastern Seared Salmon', 'content': 'Food: Middle Eastern Seared Salmon, Calories: 332.27, Protein: 19.98g, Allergens: fish', 'url': 'https://recipes.mowiscotland.co.uk/recipe/middle-eastern-seared-salmon', 'score': 0.25701842}, {'id': '6b7b3fb681be067df50ade970b35ff70320b19de7acb80c7a28c1e3427633a3f', 'name': 'Orange Salmon', 'content': 'Food: Orange Salmon, Calories: 228.31, Protein: 20.55g, Allergens: fish', 'url': 'http://healthandbeauty4ever.blogspot.com/2012/06/orange-salmon.html#!', 'score': 0.22832851}, {'id': '6d089e799567fff6edfdec60fb32191c9f3c3c77823d8368a0750cd901937e88', 'name': 'Cured Salmon', 'content': 'Food: Cured Salmon, Calories: 173.11,

In [ ]:
final_state['food_items']

[{'id': 'dacdbf3c1e3f688b3200b0484f208452e4cfad06c9b677a3fe3c08eb6116037f',
  'name': 'Middle Eastern Seared Salmon',
  'content': 'Food: Middle Eastern Seared Salmon, Calories: 332.27, Protein: 19.98g, Allergens: fish',
  'url': 'https://recipes.mowiscotland.co.uk/recipe/middle-eastern-seared-salmon',
  'score': 0.25701842},
 {'id': '6b7b3fb681be067df50ade970b35ff70320b19de7acb80c7a28c1e3427633a3f',
  'name': 'Orange Salmon',
  'content': 'Food: Orange Salmon, Calories: 228.31, Protein: 20.55g, Allergens: fish',
  'url': 'http://healthandbeauty4ever.blogspot.com/2012/06/orange-salmon.html#!',
  'score': 0.22832851},
 {'id': '6d089e799567fff6edfdec60fb32191c9f3c3c77823d8368a0750cd901937e88',
  'name': 'Cured Salmon',
  'content': 'Food: Cured Salmon, Calories: 173.11, Protein: 16.09g, Allergens: fish',
  'url': 'http://pardonmyfrenchcuisine.com/recipe/cured-salmon/',
  'score': 0.20323272},
 {'id': 'a562528f942dba04af094e2cfd4c60f316b7bf7d6b8e2a9b01ccd05ff4c248fa',
  'name': 'Whole Roa

In [ ]:
# ## SHOW PINECONE SAMPLE
# # 1. Get Index Statistics
# stats = index.describe_index_stats()
# print("--- Index Statistics ---")
# print(stats)

# # 2. Infer Schema from a Sample Record
# # Generate a dummy embedding using the loaded model
# sample_embedding = embeddings.embed_query("nutrition sample")

# # Query for 1 item to inspect metadata
# sample_result = index.query(
#     vector=sample_embedding,
#     top_k=1,
#     include_metadata=True
# )

# print("\n--- Inferred Metadata Schema ---")
# if sample_result['matches']:
#     metadata = sample_result['matches'][0]['metadata']
#     print(f"Sample Item: {sample_result['matches'][0]['id']}")
#     for key, value in metadata.items():
#         print(f"- {key}: {type(value).__name__} (e.g., {value})")
# else:
#     print("Index appears to be empty, cannot infer schema.")